# Group 5: Data Generation

After initial discussions with the client, we learned that all operational data has historically been recorded and stored in various Excel spreadsheets. To successfully migrate this data into our newly designed relational database schema, we propose a structured, multi-step process. 
* First, we will analyze the existing spreadsheets to map the available columns to the appropriate tables and fields in the new schema.
* Second, we will perform necessary data cleaning and transformations — including splitting combined fields, formatting dates, standardizing categorical variables, and handling missing values — to ensure compatibility with the database structure. Ideally we will manipulate the data prior to loading so the final import will have a smooth transition to the new database.
* Finally, we will load the cleaned datasets through csv files, matching the information that is needed to each relation in our proposed schema.

### Analyzing existing spreadshits and mapping to new schema

Following our review of the client's information, we identified 4 key spreadsheets that contain historical operational data. Each spreadsheet has multiple columns, which will be mapped, cleaned and migrated to our new schema.
1) ABC_employees.xlsx
   * first_name — Employee’s first name
   * last_name — Employee’s last name
   * department — Department (e.g., Produce, Meat, Bakery)
   * salary — Monthly salary (may require formatting)
   * hours_per_week — Contracted working hours
   * hire_date — Date of hire (some missing values)
   * location_name — Store where the employee works (to be mapped to location_id)
   * shift_date — Date of scheduled shift
   * start_time — Shift start time
   * end_time — Shift end time
   * shift_status — Scheduled, absent, or on leave
2) ABC_products_and_suppliers.xlsx
   * productname — Name of the product
   * category — Product category (e.g., Dairy, Produce, Frozen)
   * unitcost — Cost per unit (historical fluctuations noted)
   * unitprice — Retail price per unit
   * expiration_days — Expected shelf life in days
   * manufacturer_name — Manufacturer's name
   * supplier_name — Supplier’s name
   * supplier_email — Supplier’s contact email
   * supplier_phone — Supplier’s contact number
   * notes — Additional supplier notes (optional)
3) ABC_orders.xlsx
   * order_date — Date the purchase order was placed
   * expected_delivery_date — Expected delivery date
   * delivery_date — Actual delivery date (may be missing for pending orders)
   * supplier_name — Supplier for the order
   * location — Store receiving the order
   * status — Status of the order (varies, may need mapping to ('Ordered', 'Received', 'Cancelled'))
   * productname — Product ordered
   * units — Units ordered
   * unit_cost — Cost per unit at the time of order
4) ABC_sales_and_customers.xlsx
   * sale_date — Date of sale
   * location_name — Store location of the sale
   * customer_first_name — Customer first name
   * customer_last_name — Customer last name
   * age — Customer age
   * gender — Customer gender
   * email — Customer email (sometimes missing)
   * loyalty_member — Whether the customer is a loyalty member (yes/no)
   * product_name — Product sold
   * quantity — Quantity sold
   * unit_price — Price at time of sale
   * discount_applied — Discount applied during sale
   * promotion_name — Name of promotion (if any)
   * notes - Notes (including 'returns')
5) ABC_accounting.xlsx
   * expense_id	- Unique identifier for each expense
   * category - The type of expense
   * amount	- The dollar amount spent for that particular expense.
   * expense_date - The date when the expense was incurred or recorded.
   * location_name - Name the store or location associated with the expense
   * supplier_id - The ID of the supplier related to the expense, if applicable
   * description - Additional notes or a short explanation about the expense

Based from this information, we can map the data inputs from identified columns directly to our schema:
* Employees: employees and staffing
* Products and suppliers: products, manufacturers and suppliers
* Orders: purchase_orders, deliveries and purchase_order_details
* Sales and customers: customers, sales_orders, sales_orders_details and promotions

Other tables such as inventory, transactions, expenses and returns will not be manually populated from the client spreadsheets. Instead, they will be populated through an automated process and triggers. For example:
* purchase_order_details and sales_orders_details will be populated based on items from the order data.
* inventory will automatically be updated by triggers when deliveries are finalized or when products are sold.

### [Back-Up] Data generation

Since we do not have data from the client, we wrote code for an automated, randomized data generation of the 4 excel files. We used multiple iterations for the generation data, and have included a list of the final assumptions:

1) Simulate a 2-store grocery chain with operational transactions based on specific opening dates:
   * ABC Queens East: 2024-03-01
   * ABC Queens West: 2024-06-01
   * Simulation of the period ended on CURRENT_DATE, assumed late April 2025
2) Ouput excel files use different source-like column names (such as productname) and include potential data quality issues like basic 'returned' notes to simulate the client's starting point
3) Products: Fixed list of ~40 distinct products with longer shelf lives to simplify inventory turnover considerations in the simulation
4) Supplier/manufacturers: Created 30 suppliers and 30 manufacturers using Faker
5) Each product was assigned a single fixed manufacturer and suppliers (randomizing this area created duplicate manufacturers and duplicate suppliers for the same product. Although theoretically it's possible to have more than one supplier/manufacturer, this created a strong redundancy in unique product numbers)
6) Markup/Margin: To ensure an aggregagated profitability in the final analysis, base prices were set at 2~2.5x costs to make up for salary expenses. Despite these efforts, the company is still not profitable.
7) Employee count: Limited 6 employees tota across the two operational stores to control salary costs (Assumed CEO worked too)
8) Ensured 1 'Cashier' was assigned to each location, with 4 employees being random (Multi-department work is assumed but not explicitly modeled in the employees table schema because there are only 6 workers)
9) Salary: Generated yearly salary between 34,000 and 35,000 (NYC minimum of $16) prorated based on the assigned hours_per_week.
10) Staffing: Generated 3 months of schedules per employee after their hire date. Shift lengths varied depending on PT or FT. However, assumed the client didn't track this (as it was manual and not digitized).
11) Purchase orders: generated 1000 purchase orders with random units between (10,60) to lower total COGS. POs started ~45 days before opening (since stock is necessary at day of launch). Average 'Received' status was of ~75% to provide inventory for sales
12) Sales orders: Sales were constrained based on available stock from the purchase orders. Calculated total units of 'received' and created to sell a high percentage of that stock. Targeted selling 98% of calculated stock for each product, leaving a 2% inventory buffer as unsold inventory cost contributing to COGS. (If there was too much stuck, profit levels declined drastically). Sales were only generated after the opening date and had quantities between 1 and 10 units.
13) Discounts/Returns: minimized the impact by setting a low chance (1% of 5%) and a low return rate (1% of sales were 'returned'). Assumed that promotions were not really tracked well by the client.

In [95]:
# Setting up basic configurations to make sure all data comes from a pre-defined list
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta, date
import numpy as np
import os
import re

# --- Configuration & Setup ---
faker = Faker()
output_directory = '/Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/'
os.makedirs(output_directory, exist_ok=True)
print(f"Output directory set to: {output_directory}")
if not os.path.isdir(output_directory): print("Warning: Output directory does not exist or is not accessible.")

# --- Core Data Definitions ---

# 1. Locations (Operational Stores Only)
store_names = ['ABC Queens East', 'ABC Queens West']
location_start_dates = {'ABC Queens East': date(2024, 3, 1), 'ABC Queens West': date(2024, 6, 1)}
simulation_end_date = date.today()
print(f"Generating transactional data ONLY for stores: {store_names}")

# 2. Departments & Categories
departments = ['Deli', 'Cashier', 'Customer Service', 'Grocery', 'Dairy', 'Frozen', 'Beverages', 'Meat', 'Seafood', 'Cheese']
categories = ['Dairy', 'Meat', 'Seafood', 'Frozen', 'Pantry', 'Beverages', 'Cheese', 'Deli Meats']

# 3. Suppliers & Manufacturers (30 each)
defined_suppliers = list(set([faker.company() for _ in range(50)]))[:30]
manufacturers = list(set([faker.company() for _ in range(50)]))[:30]
if len(defined_suppliers) < 30 or len(manufacturers) < 30: print("Warning: Could not generate 30 unique suppliers/manufacturers.")
supplier_name_to_id = {name: i + 1 for i, name in enumerate(defined_suppliers)}
manufacturer_name_to_id = {name: i + 1 for i, name in enumerate(manufacturers)}
supplier_id_to_name = {v: k for k, v in supplier_name_to_id.items()}
manufacturer_id_to_name = {v: k for k, v in manufacturer_name_to_id.items()}
print(f"Defined {len(defined_suppliers)} suppliers, {len(manufacturers)} manufacturers.")

# 4. Unique Emails & Phones for Suppliers
supplier_contacts = {}
used_phones = set()
def clean_name_for_email(name): name = name.lower(); name = re.sub(r'[,]?\s+(llc|inc|ltd|group|and sons)$', '', name); name = re.sub(r'[^\w-]+', '', name); name = name.replace('-', ''); return name[:30]
print("Generating unique supplier emails/phones...")
for name in defined_suppliers:
    cleaned_name = clean_name_for_email(name); unique_email = f"info@{cleaned_name}.test"; sup_phone = None; phone_attempts = 0
    while phone_attempts < 10:
        sup_phone = faker.phone_number(); phone_attempts += 1
        if sup_phone not in used_phones: used_phones.add(sup_phone); break
    if phone_attempts == 10 and sup_phone in used_phones: sup_phone = f"{faker.phone_number()}-{random.randint(100,999)}"
    supplier_contacts[name] = {'email': unique_email, 'phone': sup_phone}

# 5. Products
product_definitions = [
    # Cat, BC, BP (Markup Applied), Exp, S_ID, M_ID, [SM], SC, SP
    {"name": "UHT Milk (Liter)", "cat": "Dairy", "bc": 0.80, "bp": round(0.80 * random.uniform(2.0, 2.5), 2), "exp": 180, "s_id": 1, "m_id": 1, "sm": [], "sc": None, "sp": None},
    {"name": "Parmesan Cheese (Wedge)", "cat": "Cheese", "bc": 4.00, "bp": round(4.00 * random.uniform(2.0, 2.5), 2), "exp": 120, "s_id": 2, "m_id": 2, "sm": [], "sc": None, "sp": None},
    {"name": "Sharp Cheddar Block (1lb)", "cat": "Cheese", "bc": 3.50, "bp": round(3.50 * random.uniform(2.0, 2.5), 2), "exp": 180, "s_id": 3, "m_id": 3, "sm": [], "sc": None, "sp": None},
    {"name": "Salted Butter (1lb)", "cat": "Dairy", "bc": 3.00, "bp": round(3.00 * random.uniform(2.0, 2.5), 2), "exp": 120, "s_id": 1, "m_id": 1, "sm": [], "sc": None, "sp": None},
    {"name": "Cream Cheese (8oz)", "cat": "Dairy", "bc": 1.20, "bp": round(1.20 * random.uniform(2.0, 2.5), 2), "exp": 60, "s_id": 2, "m_id": 2, "sm": [], "sc": None, "sp": None},
    {"name": "Frozen Chicken Wings (2lb)", "cat": "Frozen", "bc": 4.50, "bp": round(4.50 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 4, "m_id": 4, "sm": [1, 2, 9, 10], "sc": 4.75, "sp": round(4.75*2.5,2)},
    {"name": "Frozen Salmon Fillets (4ct)", "cat": "Frozen", "bc": 9.00, "bp": round(9.00 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 5, "m_id": 5, "sm": [], "sc": None, "sp": None},
    {"name": "Bacon (1lb)", "cat": "Meat", "bc": 3.80, "bp": round(3.80 * random.uniform(2.0, 2.5), 2), "exp": 45, "s_id": 6, "m_id": 6, "sm": [], "sc": None, "sp": None},
    {"name": "Pepperoni Slices (6oz)", "cat": "Deli Meats", "bc": 2.50, "bp": round(2.50 * random.uniform(2.0, 2.5), 2), "exp": 90, "s_id": 7, "m_id": 7, "sm": [], "sc": None, "sp": None},
    {"name": "Canned Ham (12oz)", "cat": "Meat", "bc": 2.80, "bp": round(2.80 * random.uniform(2.0, 2.5), 2), "exp": 1095, "s_id": 4, "m_id": 4, "sm": [], "sc": None, "sp": None},
    {"name": "Frozen Mixed Vegetables (1lb)", "cat": "Frozen", "bc": 1.00, "bp": round(1.00 * random.uniform(2.0, 2.5), 2), "exp": 540, "s_id": 8, "m_id": 8, "sm": [], "sc": None, "sp": None},
    {"name": "Frozen French Fries (2lb)", "cat": "Frozen", "bc": 1.80, "bp": round(1.80 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 9, "m_id": 9, "sm": [], "sc": None, "sp": None},
    {"name": "Vanilla Ice Cream (1.5qt)", "cat": "Frozen", "bc": 3.00, "bp": round(3.00 * random.uniform(2.0, 2.5), 2), "exp": 180, "s_id": 10, "m_id": 10, "sm": [6, 7, 8], "sc": 3.20, "sp": round(3.20*2.5,2)},
    {"name": "Frozen Pizza (Cheese)", "cat": "Frozen", "bc": 3.20, "bp": round(3.20 * random.uniform(2.0, 2.5), 2), "exp": 270, "s_id": 11, "m_id": 11, "sm": [], "sc": None, "sp": None},
    {"name": "Frozen Blueberries (1lb)", "cat": "Frozen", "bc": 2.80, "bp": round(2.80 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 8, "m_id": 8, "sm": [], "sc": None, "sp": None},
    {"name": "Frozen Meatballs (1lb)", "cat": "Frozen", "bc": 3.50, "bp": round(3.50 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 9, "m_id": 9, "sm": [], "sc": None, "sp": None},
    {"name": "All-Purpose Flour (5lb)", "cat": "Pantry", "bc": 1.50, "bp": round(1.50 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 12, "m_id": 12, "sm": [10, 11, 12], "sc": 1.60, "sp": round(1.60*2.5,2)},
    {"name": "Granulated Sugar (4lb)", "cat": "Pantry", "bc": 1.80, "bp": round(1.80 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 13, "m_id": 13, "sm": [10, 11, 12], "sc": 1.90, "sp": round(1.90*2.5,2)},
    {"name": "Pasta (Penne)", "cat": "Pantry", "bc": 0.85, "bp": round(0.85 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 14, "m_id": 14, "sm": [], "sc": None, "sp": None},
    {"name": "Pasta Sauce (Alfredo)", "cat": "Pantry", "bc": 1.80, "bp": round(1.80 * random.uniform(2.0, 2.5), 2), "exp": 540, "s_id": 15, "m_id": 15, "sm": [], "sc": None, "sp": None},
    {"name": "Olive Oil (Extra Virgin, 1L)", "cat": "Pantry", "bc": 6.00, "bp": round(6.00 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 16, "m_id": 16, "sm": [], "sc": None, "sp": None},
    {"name": "Canned Tuna (in Oil)", "cat": "Pantry", "bc": 0.75, "bp": round(0.75 * random.uniform(2.0, 2.5), 2), "exp": 1095, "s_id": 17, "m_id": 17, "sm": [], "sc": None, "sp": None},
    {"name": "Basmati Rice (2lb)", "cat": "Pantry", "bc": 2.00, "bp": round(2.00 * random.uniform(2.0, 2.5), 2), "exp": 1825, "s_id": 18, "m_id": 18, "sm": [], "sc": None, "sp": None},
    {"name": "Canned Chickpeas", "cat": "Pantry", "bc": 0.55, "bp": round(0.55 * random.uniform(2.0, 2.5), 2), "exp": 1095, "s_id": 19, "m_id": 19, "sm": [], "sc": None, "sp": None},
    {"name": "Peanut Butter (Crunchy)", "cat": "Pantry", "bc": 2.10, "bp": round(2.10 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 20, "m_id": 20, "sm": [], "sc": None, "sp": None},
    {"name": "Grape Jelly", "cat": "Pantry", "bc": 1.70, "bp": round(1.70 * random.uniform(2.0, 2.5), 2), "exp": 540, "s_id": 21, "m_id": 21, "sm": [], "sc": None, "sp": None},
    {"name": "Instant Coffee (Jar)", "cat": "Pantry", "bc": 3.50, "bp": round(3.50 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 22, "m_id": 22, "sm": [], "sc": None, "sp": None},
    {"name": "Crackers (Saltine)", "cat": "Pantry", "bc": 1.20, "bp": round(1.20 * random.uniform(2.0, 2.5), 2), "exp": 270, "s_id": 12, "m_id": 13, "sm": [], "sc": None, "sp": None},
    {"name": "Cereal (Oats)", "cat": "Pantry", "bc": 2.00, "bp": round(2.00 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 13, "m_id": 14, "sm": [], "sc": None, "sp": None},
    {"name": "Vegetable Oil (1 Gal)", "cat": "Pantry", "bc": 5.00, "bp": round(5.00 * random.uniform(2.0, 2.5), 2), "exp": 540, "s_id": 14, "m_id": 15, "sm": [], "sc": None, "sp": None},
    {"name": "Popcorn Kernels (30oz)", "cat": "Pantry", "bc": 2.20, "bp": round(2.20 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 15, "m_id": 16, "sm": [], "sc": None, "sp": None},
    {"name": "Cola (2 Liter)", "cat": "Beverages", "bc": 1.00, "bp": round(1.00 * random.uniform(2.0, 2.5), 2), "exp": 180, "s_id": 23, "m_id": 23, "sm": [], "sc": None, "sp": None},
    {"name": "Diet Cola (12pk Cans)", "cat": "Beverages", "bc": 4.20, "bp": round(4.20 * random.uniform(2.0, 2.5), 2), "exp": 180, "s_id": 24, "m_id": 24, "sm": [], "sc": None, "sp": None},
    {"name": "Orange Juice (with Pulp)", "cat": "Beverages", "bc": 2.60, "bp": round(2.60 * random.uniform(2.0, 2.5), 2), "exp": 45, "s_id": 25, "m_id": 25, "sm": [], "sc": None, "sp": None},
    {"name": "Sparkling Water (Lime, 8pk)", "cat": "Beverages", "bc": 3.00, "bp": round(3.00 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 26, "m_id": 26, "sm": [], "sc": None, "sp": None},
    {"name": "Iced Tea (Gallon, Sweet)", "cat": "Beverages", "bc": 2.00, "bp": round(2.00 * random.uniform(2.0, 2.5), 2), "exp": 60, "s_id": 27, "m_id": 27, "sm": [], "sc": None, "sp": None},
    {"name": "Apple Juice (64oz)", "cat": "Beverages", "bc": 2.20, "bp": round(2.20 * random.uniform(2.0, 2.5), 2), "exp": 270, "s_id": 28, "m_id": 28, "sm": [], "sc": None, "sp": None},
    {"name": "Sports Drink (Lemon-Lime)", "cat": "Beverages", "bc": 0.80, "bp": round(0.80 * random.uniform(2.0, 2.5), 2), "exp": 365, "s_id": 29, "m_id": 29, "sm": [], "sc": None, "sp": None},
    {"name": "Bottled Water (1 Liter)", "cat": "Beverages", "bc": 0.50, "bp": round(0.50 * random.uniform(2.0, 2.5), 2), "exp": 730, "s_id": 30, "m_id": 30, "sm": [], "sc": None, "sp": None},
    {"name": "Energy Drink (Regular)", "cat": "Beverages", "bc": 1.50, "bp": round(1.50 * random.uniform(2.0, 2.5), 2), "exp": 540, "s_id": 23, "m_id": 24, "sm": [], "sc": None, "sp": None},
]

# Add supplier and manufacturer names
for p in product_definitions: p['supplier_name'] = supplier_id_to_name.get(p['s_id']); p['manufacturer_name'] = manufacturer_id_to_name.get(p['m_id'])
product_lookup = {p['name']: p for p in product_definitions}
product_names = list(product_lookup.keys())
print(f"Defined {len(product_definitions)} products with EXTREME markup.")
used_s_ids = set(p['s_id'] for p in product_definitions); used_m_ids = set(p['m_id'] for p in product_definitions)
print(f"Suppliers used: {len(used_s_ids)}; Manufacturers used: {len(used_m_ids)}")

# 6. Other Configs
shift_statuses = ['Scheduled', 'Absent', 'On Leave']
hours_options = [20, 30, 35, 40]
order_statuses = ['Ordered', 'Received', 'Cancelled']
genders = ['M', 'F', 'O']
promotion_names = [None] # Effectively disable promotions for max profit
expense_categories = ['Utilities', 'Marketing', 'Rent', 'Insurance', 'Office Supplies', 'Maintenance', 'Salaries', 'Miscellaneous', 'Construction', 'Setup Costs']

print("Setup complete. Common definitions created.")

Output directory set to: /Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/
Generating transactional data ONLY for stores: ['ABC Queens East', 'ABC Queens West']
Defined 30 suppliers, 30 manufacturers.
Generating unique supplier emails/phones...
Defined 40 products with EXTREME markup.
Suppliers used: 30; Manufacturers used: 30
Setup complete. Common definitions created.


In [151]:
# Employees Table

import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta, date
import os
import sys
import math

# Ensure necessary variables from Block 1 are available
required_globals = ['faker', 'output_directory', 'departments', 'hours_options',
                    'store_names', 'shift_statuses', 'location_start_dates', 'simulation_end_date']
missing_globals = [var for var in required_globals if var not in globals() or globals().get(var) is None]
if missing_globals:
    print(f"Error: Missing required variables from Setup block (Block 1): {', '.join(missing_globals)}")
    raise NameError(f"Missing required variables from Setup block: {', '.join(missing_globals)}")
else:
    # Ensure 'Cashier' is a valid department choice
    if 'Cashier' not in departments:
        print("Error: 'Cashier' department is not in the 'departments' list defined in Block 1.")
        raise ValueError("Department 'Cashier' must be defined in Block 1.")
    # Ensure the specific store names exist
    if 'ABC Queens East' not in location_start_dates or 'ABC Queens West' not in location_start_dates:
         raise ValueError("location_start_dates dictionary in Block 1 must contain 'ABC Queens East' and 'ABC Queens West'.")


    DEFAULT_SHIFT_HOURS = (6, 8)
    NYC_MIN_WAGE = 16.00
    FULL_TIME_HOURS = 40.0
    PART_TIME_HOURS = 15.0
    BASE_SALARY_RANGE_FT = (34000.00, 36000.00)

    # --- GENERATE EXACTLY 5 EMPLOYEES WITH SPECIFIC ROLES/HIRE DATES ---
    def generate_employees(num_employees=5, num_shifts_per_week_range=(3, 5)): # Hardcoded to 5
        """Generates 5 employees with specific roles, hours, hire dates, and prorated salary."""
        employee_records = []; staffing_records = []
        print(f"Generating data for {num_employees} specific employees...")

        # Define the specific employee configurations explicitly
        employee_configs = [
            # Queens East
            {'location': 'ABC Queens East', 'hours': FULL_TIME_HOURS, 'dept_pref': None, 'hire_rule': 'random_around_start'},
            {'location': 'ABC Queens East', 'hours': PART_TIME_HOURS, 'dept_pref': None, 'hire_rule': 'random_around_start'},
            # Queens West
            {'location': 'ABC Queens West', 'hours': FULL_TIME_HOURS, 'dept_pref': 'Cashier', 'hire_rule': 'on_opening_date'}, # FT Cashier hired on opening
            {'location': 'ABC Queens West', 'hours': PART_TIME_HOURS, 'dept_pref': None, 'hire_rule': 'random_around_start'},
            {'location': 'ABC Queens West', 'hours': PART_TIME_HOURS, 'dept_pref': None, 'hire_rule': 'random_around_start'}
        ]

        if len(employee_configs) != num_employees:
             print(f"Error: Configuration list length ({len(employee_configs)}) doesn't match num_employees ({num_employees}).")
             return

        for i, config in enumerate(employee_configs):
            first_name = faker.first_name(); last_name = faker.last_name()
            hours_per_week = config['hours']
            location_name = config['location']
            loc_start_date = location_start_dates[location_name]

            # --- Assign Department ---
            if config['dept_pref'] == 'Cashier':
                department = 'Cashier'
            else:
                possible_departments = [d for d in departments if d != 'Cashier']
                if not possible_departments: possible_departments = departments
                department = random.choice(possible_departments)
            print(f"  Assigning Employee {i+1}: {department} ({hours_per_week} hrs) at {location_name}")

            # --- Generate Prorated Yearly Salary ---
            base_full_time_salary = round(random.uniform(BASE_SALARY_RANGE_FT[0], BASE_SALARY_RANGE_FT[1]), 2)
            prorated_salary = round(base_full_time_salary * (hours_per_week / FULL_TIME_HOURS), 2)
            min_wage_yearly_equivalent = round(NYC_MIN_WAGE * hours_per_week * 52, 2)
            final_yearly_salary = max(prorated_salary, min_wage_yearly_equivalent)

            # --- Determine Hire Date based on Rule ---
            if config['hire_rule'] == 'on_opening_date':
                hire_date_dt = loc_start_date
                print(f"    - Hire Date (Fixed): {hire_date_dt}")
            else: # 'random_around_start'
                # Hire between 30 days before opening and up to 60 days after opening, but not after today
                earliest_hire = loc_start_date - timedelta(days=30)
                latest_hire = min(loc_start_date + timedelta(days=60), simulation_end_date) # Hire within ~2 months of opening or by today
                # Ensure start is not after end
                if earliest_hire > latest_hire:
                    hire_date_dt = latest_hire # Hire on the latest possible day if range is invalid
                else:
                    hire_date_dt = faker.date_between(start_date=earliest_hire, end_date=latest_hire)
                print(f"    - Hire Date (Random): {hire_date_dt}")

            # 5% chance of hire date being NULL (overrides calculated date)
            hire_date = hire_date_dt if random.random() > 0.05 else None
            if hire_date is None: print("    - Hire Date set to NULL")
            # --- End Hire Date Logic ---


            # Add employee record with yearly salary
            employee_records.append({
                'first_name': first_name, 'last_name': last_name, 'department': department,
                'salary': final_yearly_salary, # Use the calculated prorated salary
                'hours_per_week': hours_per_week, 'hire_date': hire_date,
                'location_name': location_name
            })

            # Generate shifts (only after hire date and store opening)
            if hire_date is not None: # Cannot schedule shifts if hire date is unknown
                 num_shifts_total = random.randint(num_shifts_per_week_range[0], num_shifts_per_week_range[1]) * 12
                 # Shifts start only after the LATER of store opening or employee hire date
                 shift_period_start = max(loc_start_date, hire_date)
                 # Shifts end up to 14 days after simulation end date (future scheduling)
                 shift_period_end = simulation_end_date + timedelta(days=14)

                 if shift_period_start <= shift_period_end: # Only generate if there's a valid period
                     shift_hour_range = DEFAULT_SHIFT_HOURS
                     for _ in range(num_shifts_total):
                         shift_date = faker.date_between(start_date=shift_period_start, end_date=shift_period_end)
                         start_hour = random.choice([7, 8, 9, 10, 11, 12, 13, 14, 15, 16])
                         start_time = f"{start_hour:02d}:00"
                         if hours_per_week <= 20: shift_length = random.randint(max(1, shift_hour_range[0]-2), shift_hour_range[1]-2)
                         else: shift_length = random.randint(shift_hour_range[0], shift_hour_range[1])
                         shift_length = max(1, shift_length)
                         end_hour = (start_hour + shift_length) % 24; end_time = f"{end_hour:02d}:00"
                         shift_status = random.choices(shift_statuses, weights=[90, 7, 3])[0]
                         staffing_records.append({'first_name': first_name, 'last_name': last_name, 'shift_date': shift_date, 'start_time': start_time, 'end_time': end_time, 'shift_status': shift_status, 'location_name': location_name})

        df_employees = pd.DataFrame(employee_records); df_staffing = pd.DataFrame(staffing_records)
        file_path = os.path.join(output_directory, 'ABC_employees.xlsx')
        print(f"Attempting to save employee data to: {file_path}")
        try:
            df_employees['hire_date'] = pd.to_datetime(df_employees['hire_date'], errors='coerce')
            df_staffing['shift_date'] = pd.to_datetime(df_staffing['shift_date'], errors='coerce')
            with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
                df_employees.to_excel(writer, sheet_name='Employees', index=False)
                df_staffing.to_excel(writer, sheet_name='Staffing', index=False)
            print(f"Successfully generated {len(df_employees)} employees and {len(df_staffing)} shifts -> ABC_employees.xlsx")
        except Exception as e: print(f"An error occurred while saving ABC_employees.xlsx: {e}")

    # --- Execute Employee Generation ---
    print("\n--- Generating Employees (5 Specific Roles & Hire Dates) ---")
    generate_employees(num_employees=5)


--- Generating Employees (5 Specific Roles & Hire Dates) ---
Generating data for 5 specific employees...
  Assigning Employee 1: Deli (40.0 hrs) at ABC Queens East
    - Hire Date (Random): 2024-03-27
  Assigning Employee 2: Cheese (15.0 hrs) at ABC Queens East
    - Hire Date (Random): 2024-02-19
  Assigning Employee 3: Cashier (40.0 hrs) at ABC Queens West
    - Hire Date (Fixed): 2024-06-01
  Assigning Employee 4: Seafood (15.0 hrs) at ABC Queens West
    - Hire Date (Random): 2024-07-25
  Assigning Employee 5: Meat (15.0 hrs) at ABC Queens West
    - Hire Date (Random): 2024-07-04
Attempting to save employee data to: /Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_employees.xlsx
Successfully generated 5 employees and 216 shifts -> ABC_employees.xlsx


In [ ]:
# Products and Suppliers Table


# Ensure necessary variables from the setup block are available
# Required: faker, output_directory, product_lookup, product_names, supplier_contacts

def generate_products_suppliers(num_rows=500):
    """Generates product/supplier data using FIXED relationships and seasonal pricing."""
    # --- Pre-checks ---
    required_globals = ['faker', 'output_directory', 'product_lookup', 'product_names', 'supplier_contacts']
    missing_globals = [var for var in required_globals if var not in globals()]
    if missing_globals:
        print(f"Error: Missing required variables from Setup block (Block 1): {', '.join(missing_globals)}")
        print("Please run Block 1 first.")
        return
    if not product_lookup:
         print("Error: product_lookup dictionary is empty. Run Setup block (Block 1) first.")
         return
    # --- End Pre-checks ---

    data = []
    today = datetime.now().date()
    print(f"Generating {num_rows} product/supplier rows for Excel (using fixed relationships)...")

    for _ in range(num_rows):
        # Randomly select one of the 40 product NAMES
        product_name = random.choice(product_names)
        # Get the full definition for this product
        product_info = product_lookup[product_name]

        # --- Get FIXED Supplier & Manufacturer ---
        supplier_name = product_info['supplier_name']
        manufacturer_name = product_info['manufacturer_name']
        # --- End Fixed Supplier/Manufacturer ---

        # Safely get contact info
        contact_info = supplier_contacts.get(supplier_name, {'email': 'error@example.test', 'phone': 'N/A'})
        supplier_email = contact_info['email']
        supplier_phone = contact_info['phone']

        # --- Determine Cost/Price based on Season ---
        simulated_date = faker.date_between(start_date=today - timedelta(days=365), end_date=today)
        simulated_month = simulated_date.month

        current_cost = product_info['bc'] # Start with base cost
        current_price = product_info['bp'] # Start with base price

        # Check if current month is in the product's seasonal months list
        if product_info['sm'] and simulated_month in product_info['sm']:
            # Use seasonal cost/price if defined, otherwise fallback to base
            current_cost = product_info['sc'] if product_info['sc'] is not None else product_info['bc']
            current_price = product_info['sp'] if product_info['sp'] is not None else product_info['bp']
        # --- End Seasonal Cost/Price Logic ---

        # Add record to list for the Excel row
        data.append({
            "productname": product_name,
            "category": product_info['cat'],
            "unitcost": current_cost, # Cost for this specific row (potentially seasonal)
            "unitprice": current_price, # Price for this specific row (potentially seasonal)
            "expiration_days": product_info['exp'],
            "manufacturer_name": manufacturer_name, # FIXED manufacturer for this product
            "supplier_name": supplier_name, # FIXED supplier for this product
            "supplier_email": supplier_email,
            "supplier_phone": supplier_phone,
            "notes": faker.sentence() if random.random() < 0.1 else None # Less frequent notes
        })

    df = pd.DataFrame(data)
    file_path = os.path.join(output_directory, 'ABC_products_and_suppliers.xlsx')
    print(f"Attempting to save product/supplier data to: {file_path}")
    try:
        df.to_excel(file_path, index=False, engine='openpyxl')
        print(f"Successfully generated {len(df)} product/supplier rows -> ABC_products_and_suppliers.xlsx")
    except PermissionError:
        print(f"Error: Permission denied when trying to save to {file_path}. Check permissions.")
    except Exception as e:
        print(f"An error occurred while saving ABC_products_and_suppliers.xlsx: {e}")
        print(f"Type of error: {type(e)}. Check directory and dependencies.")


# --- Execute Product/Supplier Generation ---
if 'faker' in globals() and 'output_directory' in globals() and 'product_lookup' in globals():
    print("\n--- Generating Products & Suppliers (Fixed Relationships) ---")
    generate_products_suppliers(num_rows=500) # Generate 500 rows for the file
else:
    print("Skipping Product/Supplier generation: Run Setup block (Block 1) first.")

In [14]:
# Orders Table

df_orders = None # Initialize global variable

required_globals = ['faker', 'output_directory', 'defined_suppliers', 'store_names', 'order_statuses', 'product_names', 'product_lookup', 'location_start_dates', 'simulation_end_date']
missing_globals = [var for var in required_globals if var not in globals() or globals().get(var) is None]
if missing_globals: print(f"Error: Missing required variables from Setup block (Block 1): {', '.join(missing_globals)}")
else:
    # Keep moderate number of PO lines, but drastically reduce quantity per line
    def generate_orders(num_orders=1000):
        """Generates purchase order data with drastically reduced units."""
        global df_orders
        records = []; earliest_po_date = date(2024, 1, 1); latest_po_date = simulation_end_date
        print(f"Generating {num_orders} purchase order records...")
        for _ in range(num_orders):
            location = random.choice(store_names); loc_start_date = location_start_dates[location]
            po_earliest_allowed = max(earliest_po_date, loc_start_date - timedelta(days=45))
            po_latest_allowed = latest_po_date
            if po_earliest_allowed > po_latest_allowed: continue
            order_date_dt = faker.date_between(start_date=po_earliest_allowed, end_date=po_latest_allowed)

            days_to_delivery = random.randint(3, 12); expected_delivery_date = order_date_dt + timedelta(days=days_to_delivery)
            supplier_name = random.choice(defined_suppliers)
            status = random.choices(order_statuses, weights=[15, 75, 10])[0] # Even higher Received chance

            delivery_date = None
            if status == 'Received':
                delivery_delay = random.choice([-1, 0, 0, 1, 1, 2]); potential_delivery_date = expected_delivery_date + timedelta(days=delivery_delay)
                delivery_date = min(potential_delivery_date, latest_po_date); delivery_date = max(delivery_date, order_date_dt)
            elif status == 'Ordered':
                 if expected_delivery_date < latest_po_date - timedelta(days=30):
                     status = random.choice(['Received', 'Cancelled'])
                     if status == 'Received':
                        delivery_delay = random.choice([-1, 0, 0, 1, 1, 2]); potential_delivery_date = expected_delivery_date + timedelta(days=delivery_delay)
                        delivery_date = min(potential_delivery_date, latest_po_date); delivery_date = max(delivery_date, order_date_dt)

            product_name = random.choice(product_names); product_info = product_lookup[product_name]
            order_month = order_date_dt.month; unit_cost_at_order = product_info['bc']
            if product_info.get('sm') and order_month in product_info['sm']:
                unit_cost_at_order = product_info.get('sc', product_info['bc']); unit_cost_at_order = product_info['bc'] if unit_cost_at_order is None else unit_cost_at_order

            # *** DRASTICALLY REDUCED units range ***
            units = random.randint(10, 60) # Very small PO quantities

            records.append({'order_date': order_date_dt, 'expected_delivery_date': expected_delivery_date, 'delivery_date': delivery_date, 'supplier_name': supplier_name, 'location': location, 'status': status, 'productname': product_name, 'units': units, 'unit_cost': unit_cost_at_order})

        df_orders_local = pd.DataFrame(records)
        df_orders_local['order_date'] = pd.to_datetime(df_orders_local['order_date']); df_orders_local['expected_delivery_date'] = pd.to_datetime(df_orders_local['expected_delivery_date']); df_orders_local['delivery_date'] = pd.to_datetime(df_orders_local['delivery_date'])

        file_path = os.path.join(output_directory, 'ABC_orders.xlsx')
        print(f"Attempting to save order data to: {file_path}")
        try:
            df_orders_local.to_excel(file_path, index=False, engine='openpyxl')
            print(f"Successfully generated {len(df_orders_local)} order records -> ABC_orders.xlsx")
            df_orders = df_orders_local
            return df_orders
        except Exception as e: print(f"An error occurred while saving ABC_orders.xlsx: {e}"); df_orders = None; return None

    print("\n--- Generating Orders (Drastically Reduced Units) ---")
    generate_orders(num_orders=1000) # Keep 1000 PO lines, but units per line are tiny
    if df_orders is not None: print(f"Purchase order DataFrame generated with {len(df_orders)} rows.")
    else: print("Purchase order DataFrame generation failed.")



--- Generating Orders (Drastically Reduced Units) ---
Generating 1000 purchase order records...
Attempting to save order data to: /Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_orders.xlsx
Successfully generated 1000 order records -> ABC_orders.xlsx
Purchase order DataFrame generated with 1000 rows.


In [16]:
# Sales and Customers Table
# Retrieving stock from the purchase orders to make sure that all sales are within existing inventory

import math

# Ensure necessary variables from Block 1 and Block 4 are available
required_globals = ['faker', 'output_directory', 'store_names', 'genders', 'product_names',
                    'product_lookup', 'promotion_names', 'location_start_dates', 'simulation_end_date', 'df_orders']

missing_globals = [var for var in required_globals if var not in globals() or globals().get(var) is None]
if missing_globals:
    print(f"Error: Missing required variables from previous blocks: {', '.join(missing_globals)}")
    print("Please ensure Block 1 and Block 4 ran successfully and df_orders was created.")
else:
    # *** UPDATED sell_through_target default ***
    def generate_stock_based_sales(purchase_orders_df,
                                   sell_through_target=0.98, # SELL 98% of received stock
                                   max_sales_per_item=10,
                                   customer_ratio=0.7,
                                   return_ratio=0.01):
        """Generates sales data by selling a high target percentage of received stock."""

        # --- 1. Calculate Received Stock ---
        print("Calculating received stock from purchase orders...")
        if purchase_orders_df is None or purchase_orders_df.empty:
             print("Warning: Purchase orders DataFrame is empty or None. No sales generated."); return None
        received_pos = purchase_orders_df[purchase_orders_df['status'] == 'Received'][['location', 'productname', 'units']].copy()
        received_stock_agg = received_pos.groupby(['location', 'productname'])['units'].sum().reset_index()
        received_stock = {(row['location'], row['productname']): row['units'] for index, row in received_stock_agg.iterrows()}
        print(f"Calculated initial received stock for {len(received_stock)} location/product combinations.")
        if not received_stock: print("Warning: No 'Received' purchase orders found. No sales generated."); return None

        # --- 2. Generate Unique Customers ---
        # Estimate needed customers based on potential sales volume
        estimated_total_units_to_sell = sum(math.floor(v * sell_through_target) for v in received_stock.values())
        avg_sale_qty = (1 + max_sales_per_item) / 2
        num_sales_estimate = estimated_total_units_to_sell / avg_sale_qty if avg_sale_qty > 0 else 1000
        num_customers_to_generate = int(num_sales_estimate * customer_ratio * 0.3)
        customers = []; customer_emails = set(); faker.unique.clear(); attempts = 0; max_attempts = num_customers_to_generate * 3
        print(f"Generating approx {num_customers_to_generate} unique customer profiles...")
        while len(customers) < num_customers_to_generate and attempts < max_attempts:
            email = faker.unique.email(); attempts += 1
            if email not in customer_emails:
                 customers.append({'customer_first_name': faker.first_name(), 'customer_last_name': faker.last_name(), 'age': random.randint(18, 85), 'gender': random.choice(genders), 'email': email, 'loyalty_member': random.choice(['yes', 'no'])})
                 customer_emails.add(email)
        if len(customers) < num_customers_to_generate: print(f"Warning: Generated {len(customers)} unique customers.")
        if not customers: print("Warning: No customers generated, sales will be guest checkouts."); customers = [None]
        faker.unique.clear()

        # --- 3. Generate Sales Records by Iterating Through Stock ---
        print(f"Generating sales records to sell ~{sell_through_target*100:.0f}% of received stock...")
        sales_records = []
        latest_sale_date = simulation_end_date
        total_rows_generated = 0
        safety_limit = 150000 # Increased safety limit for potentially more rows

        for (location_name, product_name), total_received in received_stock.items():
            target_units_to_sell = math.floor(total_received * sell_through_target)
            units_sold_so_far = 0
            loc_start_date = location_start_dates[location_name]
            if target_units_to_sell <= 0: continue

            product_details = product_lookup.get(product_name, {"bp": 5.00, "sp": None, "sm": []})

            while units_sold_so_far < target_units_to_sell:
                qty_this_sale = random.randint(1, max_sales_per_item)
                qty_this_sale = min(qty_this_sale, target_units_to_sell - units_sold_so_far)

                sale_earliest_allowed = loc_start_date; sale_latest_allowed = latest_sale_date
                if sale_earliest_allowed > sale_latest_allowed:
                     if loc_start_date == latest_sale_date: sale_date_dt = loc_start_date
                     else: break # Stop sales for this item if date range invalid
                else: sale_date_dt = faker.date_between(start_date=sale_earliest_allowed, end_date=sale_latest_allowed)

                sale_month = sale_date_dt.month; unit_price_at_sale = product_details['bp']
                if product_details.get('sm') and sale_month in product_details['sm']:
                    unit_price_at_sale = product_details.get('sp', product_details['bp']); unit_price_at_sale = product_details['bp'] if unit_price_at_sale is None else unit_price_at_sale

                customer_info = None
                if random.random() < customer_ratio and customers[0] is not None: customer_info = random.choice(customers)

                discount_pct = 0; promotion_name = None # Keep discounts minimal/zero
                if random.random() < 0.01: discount_pct = 5 # 1% chance of 5% discount

                record = {'sale_date': sale_date_dt, 'location_name': location_name, 'customer_first_name': None if customer_info is None else customer_info['customer_first_name'], 'customer_last_name': None if customer_info is None else customer_info['customer_last_name'], 'age': None if customer_info is None else customer_info['age'], 'gender': None if customer_info is None else customer_info['gender'], 'email': None if customer_info is None else customer_info['email'], 'loyalty_member': None if customer_info is None else customer_info['loyalty_member'], 'product_name': product_name, 'quantity': qty_this_sale, 'unit_price': unit_price_at_sale, 'discount_applied': discount_pct, 'promotion_name': promotion_name, 'notes': None}
                sales_records.append(record)
                units_sold_so_far += qty_this_sale
                total_rows_generated += 1

                if total_rows_generated > safety_limit: break
            if total_rows_generated > safety_limit: print("Warning: Exceeded maximum sales row generation limit."); break

        print(f"Finished generating sales. Total records created: {total_rows_generated}")

        df_sales = pd.DataFrame(sales_records)

        # Add Returns (Minimal)
        num_returns = int(len(df_sales) * return_ratio)
        if num_returns > 0 and len(df_sales) > 0:
            returned_indices = np.random.choice(df_sales.index, size=num_returns, replace=False); df_sales.loc[returned_indices, 'notes'] = 'Returned'
            print(f"Marked {num_returns} sales records with 'Returned' note.")
        else: print("No returns marked.")

        # Save File
        file_path = os.path.join(output_directory, 'ABC_sales_and_customers.xlsx')
        print(f"Attempting to save sales/customer data to: {file_path}")
        try: df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date']); df_sales.sort_values(by=['location_name', 'product_name', 'sale_date'], inplace=True); df_sales.to_excel(file_path, index=False, engine='openpyxl'); print(f"Successfully generated {len(df_sales)} sales records -> ABC_sales_and_customers.xlsx")
        except Exception as e: print(f"An error occurred while saving ABC_sales_and_customers.xlsx: {e}")
        return df_sales

    # --- Execute Sales Generation ---
    print("\n--- Generating Sales (Selling 98% of Received Stock) ---")
    # Call the function, passing df_orders from Block 4. No revenue target needed now.
    generate_stock_based_sales(purchase_orders_df=df_orders,
                               sell_through_target=0.98, # Explicitly set 98% target
                               return_ratio=0.01)


--- Generating Sales (Selling 98% of Received Stock) ---
Calculating received stock from purchase orders...
Calculated initial received stock for 80 location/product combinations.
Generating approx 1071 unique customer profiles...
Generating sales records to sell ~98% of received stock...
Finished generating sales. Total records created: 5127
Marked 51 sales records with 'Returned' note.
Attempting to save sales/customer data to: /Users/kimminsung/Desktop/Columbia/SQL/SQL Final Project/ABC_sales_and_customers.xlsx
Successfully generated 5127 sales records -> ABC_sales_and_customers.xlsx


This concludes the succesful data migration from the client to the newly formed schema.